# MESSAGEix-Pakistan 
### Baseline Model
In this notebook, we are reading data and building baseline scenerio.

<img src="https://wit.lums.edu.pk/sites/default/files/inline-images/WIT_Banner.jpg" alt="Girl in a jacket" width="850" height="250">

In [1]:
# fundamental libraries
import os
import pandas as pd
import numpy as np
import ixmp
import message_ix
from message_ix import log

# for reporting and visualization
from modelFiles.plotter_pakistan import plot, plotter

# script functions
from modelFiles.update_units_pakistan import unit_correction
from datetime import datetime
import time

# autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all
%reload_ext autoreload
%matplotlib inline

In [2]:
# saving current working directory path for later repeated use
cwd_path = os.getcwd()

Create scenario

In [3]:
# creating ixmp platform object
new_mp = ixmp.Platform()

# creating a new, empty scenario object
scenario = message_ix.Scenario(
    new_mp, model="COMMITTED", scenario="baseline", version="new"
)

2025-04-29 11:41:13,283  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2025-04-29 11:41:13,287  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/awais/.local/share/ixmp/localdb/default' (user: ixmp)...


Read Data

In [4]:
# loading data (sets & parameters) into our model - latest data file is MESSAGEix_Pakistan_SSP2_V2.2_Baseline.xlsx
data_path = "/Users/awais/Documents/GitHub/NEST-Pakistan/model/scenarios/baseline/baseline.xlsx"
scenario.read_excel(data_path, add_units=True, commit_steps=False, init_items=True,)

In [28]:
scenario = scenario.clone(scenario.model, "bau_2025", keep_solution=False, shift_first_model_year=2025)

at.ac.iiasa.ixmp.exceptions.IxException: at.ac.iiasa.ixmp.exceptions.IxException: There was a problem reading the blob of parameter 'resource_volume' from the database!

In [20]:
scenario.check_out()

In [21]:
# keep the model's temporal scope till 2060 only
years = scenario.set("year")
remove_years = [x for x in years if x >= 2070]
scenario.remove_set("year", remove_years)
years = scenario.set("year")

In [22]:
lastmodelyr = pd.DataFrame(
    {
        "type_year": ["lastmodelyear"],
        "year": [2070]
    }
)
scenario.add_set("cat_year", lastmodelyr)

ValueError: The index set 'year' does not have an element '2070'!

In [9]:
# # change first model year from 2020 to 2025
# model_horizon = scenario.set("cat_year")
# firstmodelyr = model_horizon.loc[model_horizon["type_year"] == "firstmodelyear"]
# scenario.remove_set("cat_year", firstmodelyr)
# firstmodelyr = pd.DataFrame(
#     {
#         "type_year": ["firstmodelyear"],
#         "year": [2025]
#     }
# )
# scenario.add_set("cat_year", firstmodelyr)
# new_horizon = scenario.set("cat_year")
# print(new_horizon)

In [10]:
# bound_activity_up_2020 = scenario.par('bound_activity_up', {'year_act': 2020})
# bound_new_capacity_up_2020 = scenario.par('bound_new_capacity_up', {'year_vtg': 2020})

# bound_activity_up_2020_u = bound_activity_up_2020[np.isfinite(bound_activity_up_2020['value'])]
# bound_new_capacity_up_2020_u = bound_new_capacity_up_2020[np.isfinite(bound_new_capacity_up_2020['value'])]

In [11]:
# hist_act = bound_activity_up_2020_u.copy()
# hist_act['unit'] = 'GWa'
# scenario.add_par('historical_activity', hist_act)

In [12]:
# hist_cap = bound_new_capacity_up_2020_u.copy()
# hist_cap['unit'] = 'GW'
# scenario.add_par('historical_new_capacity', hist_cap)


##### Solve the Model

In [9]:
log.info(f"version number before commit(): {scenario.version}")

# commit the model structure and input data (sets and parameters)
# scenario.commit(comment="Add all data from excel file to scenario")
scenario.set_as_default()

# exporting the built model (Scenario) to GAMS with an optional case name
caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)

# solve model
scenario.solve(case=caseName)

scenario.var("OBJ")["lvl"]

--- Warning: The GAMS version [49.3.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 04/29/25 11:00:26 49.3.0 7de46a92 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/49/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/awais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/awais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/49/Resources/
    LogOption 4
    --in /Users/awais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_COMMITTED__baseline__v1.gdx
    --out /Users/awais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_COMMITTED__baseline__v1.gdx
    --iter /Users/awais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_COMMITTED__baseline__v1.gdx
Licensee: Small MUD - 5 User License                     S240905|0002AP-GEN
          IIASA, Information and Communication Technol

2025-04-29 11:00:29,139 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-04-29 11:00:29,142 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


23709.583984375

In [ ]:
from model.report.legacy.iamc_report_hackathon import report
timestamp = f"{str(datetime.now().strftime('%Y-%m-%d--%H-%M'))}"
start = time.time()
df, path_name= report(mp=new_mp, scen=scenario, out_dir="reporting_outputs", out_file_timestamp = timestamp, IDEA_format=False)
end = time.time()

In [10]:
scenario.set("cat_year")

,type_year,year
0,1950,1950
1,1955,1955
2,1960,1960
3,1965,1965
4,1970,1970
5,1975,1975
6,1980,1980
7,1985,1985
8,1990,1990
9,1995,1995


##### Plotting Results

In [15]:
path = r"output"

alldf = plotter(scenario, caseName, path)

plot(alldf, caseName, path)

Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.62x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)
Axes(0.125,0.11;0.465x0.462)


In [16]:
# close the connection to the database
new_mp.close_db() 